<a href="https://colab.research.google.com/github/mayait/CursoAnalisisDatos_IA_2026/blob/main/sitio/labs/lab_11.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Abrir en Colab"/></a>

# Laboratorio 11 · El mapa de los modelos y la ecuación de valor

Semana comprimida en una sola sesión de noventa minutos y bisagra del curso: aquí termina la parte
descriptiva y empieza la predictiva. **Casi no hay código, y eso es deliberado.** Hoy se instalan tres
cosas que deciden si los modelos de las semanas 12 a 14 sirven para algo: **la taxonomía de familias**,
para decir en qué categoría cae un problema antes de tocar un algoritmo; **la línea base obligatoria**,
el número contra el que se juzga cualquier modelo y que casi nadie calcula; y **la ecuación de valor**,
que estima cuánto vale resolver un problema **antes** de intentar resolverlo.

Hoy no se ajusta ningún modelo de verdad. `LinearRegression` llega la semana 12, y `LogisticRegression`,
el árbol de decisión y la variable de abandono con su ventana de observación llegan la semana 14. Lo
único que se ejecuta hoy son dos líneas base, que es contra lo que esos modelos van a competir. Al final
del cuaderno vas a tener un catálogo de doce problemas de negocio clasificados, y al menos dos de ellos
no van a necesitar ningún modelo.

> **Hoy haces** · Clasificas problemas de negocio en cinco familias y nombras la decisión que cada
> una habilita (90 min, sesión única). Calculas en una sola tabla la línea base de un problema de
> pronóstico y de uno de propensión con `DummyRegressor` y `DummyClassifier`, después de apartar el
> conjunto de prueba con `train_test_split`. Traduces la ecuación de valor a una función de Python y la
> aplicas a una cartera de tres proyectos. Cierras completando el catálogo de doce problemas de
> Comercial Andina, tres de ellos ya resueltos como ejemplo.
>
> **Entrega** · Evaluación intermedia de criterio de negocio. Este cuaderno ejecutado, el catálogo de
> doce problemas completo con familia, decisión, línea base y valor anual estimado, la identificación
> de **cuáles no justifican ningún modelo** con su explicación, y el mismo análisis aplicado al problema
> del proyecto propio.
> Nombre de archivo: `lab_11_apellido.ipynb`.

In [ ]:
# --- Setup del entorno ---
from pathlib import Path
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

sns.set_theme(style="whitegrid", palette="deep")
plt.rcParams["figure.figsize"] = (10, 4)
pd.set_option("display.max_columns", 40)
pd.set_option("display.float_format", lambda v: f"{v:,.2f}")

# Los datos de Comercial Andina viven en sitio/datos/
REPO = "https://github.com/mayait/CursoAnalisisDatos_IA_2026.git"
COPIA = Path("/content/CursoAnalisisDatos_IA_2026")
CANDIDATOS = [Path("../datos"), Path("datos"), Path("sitio/datos"),
              COPIA / "sitio" / "datos"]
DATOS = next((p for p in CANDIDATOS if p.exists()), None)
if DATOS is None:
    # En Colab el cuaderno llega solo: se trae el repositorio una sola vez.
    import subprocess
    subprocess.run(["git", "clone", "--depth", "1", REPO, str(COPIA)], check=True)
    DATOS = COPIA / "sitio" / "datos"

print("Setup completo ✓")
print(f"pandas {pd.__version__} · datos en {DATOS.resolve()}")

## 1. Cinco familias y una pregunta por familia

Casi todo lo que una empresa pide cabe en cinco familias. Lo que las separa no es el algoritmo —hay
docenas para cada una— sino **qué forma tiene la respuesta y qué decisión permite tomar**.

Aprender la tabla de memoria no sirve de nada. Lo que sirve es el ejercicio inverso: te dan un problema
y tienes que decir en qué fila cae, cuál es la decisión concreta y qué pasa si el modelo se equivoca.

In [ ]:
familias = pd.DataFrame([
    ("Pronóstico", "un número", "¿cuánto?",
     "cuántas cajas de aceite pido para Cuenca en diciembre",
     "pedir de más o de menos", "la media del mismo mes del año pasado"),
    ("Propensión", "una probabilidad entre 0 y 1", "¿quién?",
     "qué clientes van a dejar de comprar en los próximos 90 días",
     "llamar a quien no hacía falta o no llamar a quien se iba",
     "la tasa histórica: uno de cada tres no vuelve"),
    ("Segmentación", "una etiqueta de grupo", "¿en qué grupos se parte esto?",
     "qué grupos de clientes reciben trato distinto",
     "diseñar una campaña para un grupo que no existe", "un solo grupo: tratar a todos igual"),
    ("Asociación", "una regla si-entonces", "¿qué va con qué?",
     "qué productos se compran juntos y hay que poner cerca",
     "cambiar el planograma sin efecto", "el producto más vendido para todos"),
    ("Optimización", "una asignación de recursos", "¿cuánto de cada cosa?",
     "cómo reparto el presupuesto de marketing entre radio, digital y volantes",
     "gastar donde rinde menos", "repartir igual, o como el año pasado"),
], columns=["familia", "qué devuelve", "pregunta que contesta", "ejemplo en Comercial Andina",
            "qué pasa si se equivoca", "línea base típica"])

print(familias[["familia", "qué devuelve", "pregunta que contesta"]].to_string(index=False))
print("\n" + familias[["familia", "ejemplo en Comercial Andina", "línea base típica"]]
      .to_string(index=False))

📌 **La columna que importa es la última.** Cada familia trae de fábrica una regla ingenua que
contesta la misma pregunta sin aprender nada, y esa regla es el rival del modelo. Un pronóstico que no
le gana a «lo mismo que el año pasado» no es un pronóstico: es un gasto.

Y hay una sexta categoría que no está en la tabla y es la más frecuente: **ninguna**. Muchas preguntas
no necesitan un modelo porque ya tienen respuesta exacta —«¿cuánto facturamos el mes pasado?» es un
`groupby`, no una predicción— o porque lo que falta no es un algoritmo sino un dato, un experimento o
una decisión que alguien no quiere tomar. Reconocer esa sexta categoría es la mitad del criterio que
esta semana evalúa.

## 2. La línea base obligatoria

Antes de la línea base hay un paso que no se salta: **apartar el conjunto de prueba**. Se aparta al
principio, antes de mirar nada, y no se vuelve a tocar hasta el final. Si eliges variables, umbrales o
algoritmos mirando los datos de prueba, ese conjunto ya no mide nada: te devuelve tu propia información.

Dos problemas de Comercial Andina, uno de cada tipo, con los datos tal como vienen —hoy no se construye
ninguna variable objetivo:

- **Pronóstico** · ¿cuánto va a valer esta factura? La línea base predice siempre lo mismo: la media o
  la mediana histórica.
- **Propensión** · ¿esta venta va a terminar en devolución? La línea base predice siempre la clase
  mayoritaria, o tira una moneda cargada con la proporción histórica.

`DummyRegressor` y `DummyClassifier` hacen exactamente eso y nada más: **no miran ninguna variable**.
Por eso les pasamos una columna de ceros como predictor, para que quede a la vista.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.dummy import DummyClassifier, DummyRegressor

ventas = pd.read_csv(DATOS / "ventas_limpias.csv", parse_dates=["fecha"])
productos = pd.read_csv(DATOS / "productos.csv")

v = ventas.merge(productos[["producto_id", "costo_unitario"]], on="producto_id",
                 how="left", validate="m:1")
v["monto"] = v["cantidad"] * v["precio_unitario"] * (1 - v["descuento"])
v["margen"] = v["monto"] - v["cantidad"] * v["costo_unitario"]
compras = v[~v["es_devolucion"]]

# Problema de PRONÓSTICO: el monto de cada factura (columnas que ya existen, nada que construir)
facturas = compras.groupby("factura_id")["monto"].sum()
# Problema de PROPENSIÓN: la línea de venta terminó en devolución (columna del archivo)
devuelta = v["es_devolucion"].astype(int)

# El Dummy no mira ninguna variable. Le pasamos una columna de ceros para dejarlo explícito.
CEROS_R = np.zeros((len(facturas), 1))
CEROS_C = np.zeros((len(devuelta), 1))

Xr_ent, Xr_pru, yr_ent, yr_pru = train_test_split(
    CEROS_R, facturas.values, test_size=0.25, random_state=SEED)
Xc_ent, Xc_pru, yc_ent, yc_pru = train_test_split(
    CEROS_C, devuelta.values, test_size=0.25, random_state=SEED, stratify=devuelta)

print(f"pronóstico · {len(facturas):,} facturas  → prueba apartada: {len(yr_pru):,}")
print(f"propensión · {len(devuelta):,} líneas de venta → prueba apartada: {len(yc_pru):,}")
print(f"devoluciones en el archivo: {devuelta.mean():.2%}  ← clase minoritaria")
print("\nEl conjunto de prueba no se vuelve a tocar hasta el final.")

In [ ]:
filas = []
for estrategia, etiqueta in [("mean", "predecir siempre la media histórica"),
                             ("median", "predecir siempre la mediana histórica")]:
    d = DummyRegressor(strategy=estrategia).fit(Xr_ent, yr_ent)
    p = d.predict(Xr_pru)
    filas.append({"problema": "pronóstico · ¿cuánto vale esta factura?", "línea base": etiqueta,
                  "qué contesta siempre": f"{p[0]:,.2f} dólares",
                  "métrica": "error absoluto medio", "valor": np.abs(yr_pru - p).mean()})

for estrategia, etiqueta in [("most_frequent", "predecir siempre la clase mayoritaria"),
                             ("stratified", "una moneda cargada con la proporción histórica")]:
    d = DummyClassifier(strategy=estrategia, random_state=SEED).fit(Xc_ent, yc_ent)
    p = d.predict(Xc_pru)
    filas.append({"problema": "propensión · ¿esta venta se devuelve?", "línea base": etiqueta,
                  "qué contesta siempre": "no se devuelve" if estrategia == "most_frequent"
                                          else "cara o cruz cargada",
                  "métrica": "exactitud", "valor": (p == yc_pru).mean()})
    if estrategia == "most_frequent":
        RECALL_BASE = p[yc_pru == 1].sum() / (yc_pru == 1).sum()

lineas_base = pd.DataFrame(filas)
print("LAS DOS LÍNEAS BASE, EN UNA SOLA TABLA · ninguna mira una sola variable\n")
print(lineas_base.to_string(index=False, float_format=lambda x: f"{x:,.4f}"))
print(f"\nDe las devoluciones reales del conjunto de prueba, la línea base mayoritaria "
      f"encuentra {RECALL_BASE:.0%}.")
print(f"Ticket medio {facturas.mean():,.2f} · mediana {facturas.median():,.2f} "
      f"· razón {facturas.mean() / facturas.median():.2f}  ← la bimodalidad de la semana 3")

In [ ]:
# ✅ Comprobación 1 · si algo de esto falla, tu tabla de líneas base no es la del cuaderno
mae_media = lineas_base.loc[0, "valor"]
exactitud_mayoritaria = lineas_base.loc[2, "valor"]

assert abs(facturas.mean() - 180.49) < 0.5, \
    "Revisa el filtro de devoluciones: tu ticket medio no coincide con el del curso (180,49)"
assert abs(mae_media - 213.75) < 1.0, \
    "El error de la línea base de la media no coincide: ¿usaste ventas_limpias.csv y SEED = 42?"
assert abs(exactitud_mayoritaria - 0.9765) < 0.005, \
    "La exactitud de la línea base mayoritaria no coincide: revisa el stratify del train_test_split"
assert RECALL_BASE == 0, \
    "La línea base mayoritaria no puede encontrar ninguna devolución: revisa cómo calculaste el recall"

print("Comprobación 1 superada ✓  las dos líneas base son las del cuaderno")

⚠️ **Una línea base que dice «esta venta no se devuelve» acierta el 97,65 % de las veces, y no sirve
para nada.** De las devoluciones reales encuentra cero. Ese es el número contra el que hay que comparar
cualquier cosa que se construya después, y es la razón por la que la exactitud sola no significa nada.

En el pronóstico pasa lo mismo con otra cara. La línea base de la media falla por **213,75 dólares** en
una factura cuyo ticket medio es 180,49; la de la mediana falla por **165,10** y acierta más, porque el
ticket es bimodal y la media está tirada por los mayoristas. Las dos son legítimas y hay que **declarar
cuál se usa antes de empezar**, porque el modelo de la semana 12 se comparará contra la que elijas.

**Primer resultado de negocio del laboratorio: sin variables que separen mayoristas de minoristas,
ningún modelo de pronóstico de facturas va a servir para planificar nada.**

Y aquí paramos: lo que hoy queda escrito en el cuaderno es **contra qué van a competir** los modelos de
las semanas 12 y 14.

## 3. La ecuación de valor

Aquí se decide si el proyecto existe. La ecuación tiene cuatro términos y ninguno es técnico:

> **valor anual = decisiones al año × mejora esperada × valor unitario − costo de construir − costo de mantener**

- **Decisiones al año** · cuántas veces alguien va a hacer algo distinto por culpa del modelo. No
  cuántos datos hay.
- **Mejora esperada** · cuánto mejor es la decisión con el modelo que con la línea base. En puntos.
- **Valor unitario** · cuánto vale en dinero acertar una vez.
- **Costos** · construirlo (una vez, repartido entre los años de vida útil) y mantenerlo (todos los
  años, y siempre se subestima).

La escribimos como función de Python porque se va a usar doce veces esta semana y otras tantas en el
proyecto.

In [ ]:
HOY = compras["fecha"].max()
ANIOS = ((HOY.to_period("M") - compras["fecha"].min().to_period("M")).n + 1) / 12
MARGEN_ANUAL_CLIENTE = v.groupby("cliente_id")["margen"].sum().mean() / ANIOS
MARGEN_ANUAL_TOTAL = v["margen"].sum() / ANIOS
COMBINACIONES = productos["producto_id"].nunique() * ventas["sucursal_id"].nunique()

print(f"margen anual de Comercial Andina           : {MARGEN_ANUAL_TOTAL:,.2f}")
print(f"margen anual medio por cliente             : {MARGEN_ANUAL_CLIENTE:,.2f}")
print(f"combinaciones producto × sucursal          : {COMBINACIONES:,}\n")


def valor_anual(decisiones_al_anio, mejora_esperada, valor_unitario,
                costo_construir=0.0, costo_mantener_anio=0.0, vida_util_anios=3):
    """La ecuación de valor del curso. Devuelve el desglose, no solo el número."""
    beneficio = decisiones_al_anio * mejora_esperada * valor_unitario
    costo = costo_construir / vida_util_anios + costo_mantener_anio
    return pd.Series({
        "beneficio anual": beneficio,
        "costo anual": costo,
        "valor anual": beneficio - costo,
        "¿se hace?": "sí" if beneficio - costo > 0 else "no",
    })


cartera = pd.DataFrame([
    ("Propensión al abandono", 300, 0.12, MARGEN_ANUAL_CLIENTE, 9000, 3600),
    ("Pronóstico de demanda por SKU y tienda", COMBINACIONES * 12, 0.04,
     MARGEN_ANUAL_TOTAL / (COMBINACIONES * 12), 14000, 6000),
    ("Reporte del total facturado del mes", 12, 0.0, MARGEN_ANUAL_CLIENTE, 4000, 1200),
], columns=["proyecto", "decisiones", "mejora", "valor unitario", "construir", "mantener"])

evaluacion = cartera.apply(
    lambda r: valor_anual(r["decisiones"], r["mejora"], r["valor unitario"],
                          r["construir"], r["mantener"]), axis=1)
cartera = pd.concat([cartera, evaluacion], axis=1).set_index("proyecto")
print(cartera.round(2).to_string())

# ¿Cuánto tiene que mejorar el modelo de abandono para empatar?
equilibrio = (9000 / 3 + 3600) / (300 * MARGEN_ANUAL_CLIENTE)
print(f"\npunto de equilibrio del modelo de abandono: hace falta retener "
      f"{equilibrio:.2%} más de los 300 contactados para que el proyecto no pierda dinero")

📌 **El pronóstico de demanda vale 9 133,66 al año y el modelo de abandono 3 577,21: dos veces y media
más**, y no porque sea mejor modelo. Es porque se usa 5 328 veces al año en lugar de 300. La ecuación
de valor casi siempre la gana el término de la izquierda —**cuántas veces se decide**— y casi nunca la
precisión del algoritmo. Un modelo mediocre que se usa todos los días vale más que uno excelente que se
usa cada trimestre. Cada decisión de pedido vale 92,91 dólares de margen y el modelo captura el 4 %:
tres dólares con setenta por decisión, multiplicados por 5 328 decisiones.

El tercer proyecto no es malo: **es que no es un proyecto de modelo.** Su mejora esperada es cero
porque la respuesta exacta ya existe con un `groupby`, y por eso la ecuación devuelve el costo con
signo negativo.

El punto de equilibrio es la cifra que se lleva a la reunión: con presupuesto para 300 llamadas, el
modelo de abandono necesita retener un **7,78 % adicional** de los contactados para no perder dinero.
Si el equipo comercial dice que su tasa de rescate es del 12 %, el proyecto se aprueba; si dice que no
la ha medido nunca, el proyecto se aplaza y lo que se aprueba es medirla.

## 4. El catálogo de doce problemas

El entregable de la semana. Doce problemas reales de Comercial Andina; tres vienen resueltos como
ejemplo y los otros nueve se completan en equipo. Las cuatro columnas se llenan **en este orden**, y el
orden importa: si no puedes escribir la decisión, no hay familia que valga.

- **Familia** · una de las cinco, o **«Ninguna»**. Escribir «Ninguna» no es rendirse: es la respuesta
  correcta en la mitad de los casos reales y la más difícil de defender ante alguien que ya decidió que
  quiere un modelo.
- **Decisión que habilita** · un verbo y un sujeto. *«Se decide a quiénes llama el equipo comercial el
  lunes»* vale; *«mejorar la retención»* no, porque no dice quién hace qué.
- **Línea base** · qué se hace hoy sin modelo, con detalle suficiente para poder calcularlo, como las
  dos de la sección 2. Si no existe línea base, el problema no está bien planteado.
- **Valor anual estimado** · el resultado de `valor_anual()`, con los supuestos escritos al lado.

`verificar_catalogo()` comprueba las tres condiciones de entrega, y la tercera es la interesante: **el
catálogo no se acepta si todos los problemas necesitan un modelo.**

In [ ]:
catalogo = pd.DataFrame([
    (1, "¿Cuánto vamos a facturar el mes que viene en cada sucursal?",
     "Pronóstico", "cuánto inventario y personal se programa para el mes",
     "la facturación del mismo mes del año pasado", 18000),
    (2, "¿Qué clientes van a dejar de comprar en los próximos 90 días?",
     "Propensión", "a quiénes llama el equipo comercial esta semana",
     "llamar a los que llevan más días sin comprar", 9800),
    (3, "¿A quién le mando la campaña de reactivación si solo alcanza para 300 clientes?",
     "", "", "", np.nan),
    (4, "¿Qué productos se compran juntos y deberían estar cerca en la góndola?",
     "", "", "", np.nan),
    (5, "¿Cuánto facturamos el mes pasado?",
     "Ninguna", "ninguna: es un reporte, la respuesta exacta ya existe",
     "un groupby sobre ventas.csv", 0),
    (6, "¿Cuántas cajas abrimos el sábado por la tarde en Quito?",
     "", "", "", np.nan),
    (7, "¿Qué precio le pongo a un producto que todavía no hemos vendido nunca?",
     "", "", "", np.nan),
    (8, "¿Cuánto pido de cada producto para cada tienda este mes?",
     "", "", "", np.nan),
    (9, "¿Cuál de dos diseños del correo de reactivación convierte más?",
     "", "", "", np.nan),
    (10, "¿Cómo reparto los 60 000 dólares de marketing entre radio, digital y volantes?",
     "", "", "", np.nan),
    (11, "¿Le doy 10 % de descuento al cliente que amenaza con irse?",
     "", "", "", np.nan),
    (12, "¿En qué ciudad conviene abrir la sexta tienda?",
     "", "", "", np.nan),
], columns=["n", "problema", "familia", "decisión que habilita", "línea base",
            "valor anual estimado"])

pd.set_option("display.max_colwidth", 60)
print(catalogo.to_string(index=False))
print(f"\nresueltos como ejemplo: {int(catalogo['familia'].ne('').sum())} de {len(catalogo)}")


def verificar_catalogo(df, minimo_sin_modelo=2):
    completas = df[df["familia"].ne("")]
    sin_modelo = completas[completas["familia"] == "Ninguna"]
    validas = {"Pronóstico", "Propensión", "Segmentación", "Asociación", "Optimización", "Ninguna"}
    desconocidas = set(completas["familia"]) - validas

    print(f"1 · filas completas          : {len(completas)} de {len(df)}"
          f"   {'OK' if len(completas) == len(df) else '← faltan ' + str(len(df) - len(completas))}")
    print(f"2 · familias válidas         : {'OK' if not desconocidas else '← revisar ' + str(desconocidas)}")
    print(f"3 · problemas sin modelo     : {len(sin_modelo)} "
          f"{'OK' if len(sin_modelo) >= minimo_sin_modelo else f'← hacen falta al menos {minimo_sin_modelo}'}")
    if len(sin_modelo):
        print("    " + " · ".join(f"#{n}" for n in sin_modelo["n"]))
    listo = len(completas) == len(df) and not desconocidas and len(sin_modelo) >= minimo_sin_modelo
    print(f"\n{'CATÁLOGO LISTO PARA ENTREGAR' if listo else 'CATÁLOGO INCOMPLETO: sigue trabajando'}")
    return listo


verificar_catalogo(catalogo)

In [ ]:
sin_modelo = pd.DataFrame([
    ("#5 · ¿cuánto facturamos el mes pasado?", 12, 0.0, MARGEN_ANUAL_CLIENTE, 4000, 1200,
     "la respuesta exacta ya existe: un modelo solo puede empeorarla"),
    ("#9 · ¿qué diseño de correo convierte más?", 2, 0.0, MARGEN_ANUAL_CLIENTE, 7000, 2400,
     "se contesta con el experimento A/B de la semana 9, no con un modelo"),
], columns=["problema", "decisiones", "mejora", "valor unitario", "construir", "mantener", "por qué"])

resultado = sin_modelo.apply(
    lambda r: valor_anual(r["decisiones"], r["mejora"], r["valor unitario"],
                          r["construir"], r["mantener"]), axis=1)
demostracion = pd.concat([sin_modelo[["problema", "decisiones", "mejora"]], resultado,
                          sin_modelo[["por qué"]]], axis=1)
print(demostracion.round(2).to_string(index=False))
print("\nLos dos dan valor negativo por la misma razón: la mejora esperada sobre la línea base es 0.")
print("No es que el modelo sea malo. Es que no hay nada que mejorar.")

# Demostrado, se escribe en el catálogo. Este paso es el único que autoriza a poner «Ninguna».
catalogo.loc[catalogo["n"] == 9, ["familia", "decisión que habilita", "línea base",
                                  "valor anual estimado"]] = [
    "Ninguna", "ninguna: se decide con un experimento A/B, no con un modelo",
    "el diseño actual, medido durante 30 días", 0]

print("")
print("Con la fila #9 clasificada, el catálogo ya cumple la condición 3:")
verificar_catalogo(catalogo)


# ✅ Comprobación 2 · el catálogo tiene que llegar a la entrega con estas tres cosas
assert demostracion.loc[0, "valor anual"] < 0, \
    "El problema #5 tiene que dar valor anual negativo: revisa que su mejora esperada sea 0"
assert (catalogo["familia"] == "Ninguna").sum() >= 2, \
    "Hacen falta al menos dos problemas clasificados como «Ninguna»: revisa la celda anterior"
assert catalogo.loc[catalogo["n"] == 9, "familia"].iloc[0] == "Ninguna", \
    "La fila #9 no quedó reclasificada: vuelve a ejecutar la celda desde el principio"
assert abs(equilibrio - 0.0778) < 0.002, \
    "El punto de equilibrio no coincide: revisa los costos de construir y mantener"

print("\nComprobación 2 superada ✓  quedan ocho filas por llenar y son tuyas")

### 🌶️ Ejercicio 1 — Guiado

Completa las ocho filas que faltan del catálogo. Para cada una: familia (o «Ninguna»), decisión con
verbo y sujeto, línea base calculable y valor anual con `valor_anual()`. Después vuelve a correr
`verificar_catalogo()` hasta que imprima **CATÁLOGO LISTO PARA ENTREGAR**.

In [ ]:
# TU CÓDIGO AQUÍ
# Pista 1: catalogo.loc[catalogo["n"] == 3, "familia"] = "Propensión"  (y así con las demás columnas)
# Pista 2: los problemas #7 y #12 no tienen historial en la base: mira si el dato existe ANTES
#          de asignarles una familia. Sin datos no hay modelo, por muy clara que sea la familia
# Pista 3: para el valor anual usa valor_anual() con supuestos escritos en un comentario al lado.
#          Si no sabes cuántas decisiones al año son, ese es el primer dato que hay que ir a buscar
# Pista 4: el problema #11 se parece a un modelo y es una regla de tres líneas. Compruébalo con la
#          ecuación: ¿cuánto vale acertar una vez, y cuántas veces al año pasa?

### 🔥 Desafío

Aplica el análisis completo **al problema del proyecto de tu grupo**, con las cifras reales de la
empresa del caso. Necesitas: (a) la familia y por qué no es ninguna de las otras cuatro, (b) la decisión
concreta con nombre y apellido de quien la toma, (c) la línea base calculada —no descrita: calculada—
con `DummyClassifier` o `DummyRegressor` sobre los datos del caso, (d) la ecuación de valor con los
cinco supuestos declarados, y (e) el punto de equilibrio: qué mejora mínima hace que el proyecto no
pierda dinero. Si el resultado es negativo, esa es la entrega: un «no» bien argumentado vale igual que
un «sí».

In [ ]:
# TU CÓDIGO AQUÍ
# Pista 1: la línea base se calcula con train_test_split y el Dummy que corresponda,
#          igual que en la sección 2. Si el caso no tiene datos suficientes, dilo con el número
# Pista 2: el punto de equilibrio es (costo_construir / vida + costo_mantener) /
#          (decisiones_al_anio * valor_unitario)
# Pista 3: los cinco supuestos son decisiones al año, mejora esperada, valor unitario,
#          costo de construir y costo de mantener. Cada uno con su fuente

### 🎯 Reto en clase (15 min)

En equipos y contra reloj: **el juicio de los doce problemas**. Cada equipo defiende que tres de los
doce problemas del catálogo NO necesitan modelo, y el equipo de al lado defiende que sí. Gana el
argumento que traiga la ecuación de valor con números, no el que hable mejor. Regla del torneo: quien
diga «con machine learning se podría» sin poder decir cuántas decisiones al año cambia, pierde el punto
automáticamente.

In [ ]:
# TU CÓDIGO AQUÍ
# Pista: prepara el argumento con la ecuación, no con adjetivos. El molde es siempre el mismo:
#   valor_anual(decisiones_al_anio=<cuántas veces al año>,
#               mejora_esperada=<cuánto mejor que la línea base>,
#               valor_unitario=<cuánto vale acertar una vez>,
#               costo_construir=<una vez>, costo_mantener_anio=<todos los años>)
# El punto se gana enseñando qué supuesto habría que cambiar para que el signo se invierta

## La trampa de hoy

⚠️ **Elegir el algoritmo antes que la decisión.** Se reconoce siempre por la misma frase: *«queremos
aplicar redes neuronales»*, dicha por alguien que no puede terminar la oración *«y entonces el lunes
haremos distinto…»*.

La demostración usa el mismo modelo de abandono de la sección 3, sin cambiarle ni una línea. Lo único
que cambia entre las dos columnas es si hay una decisión detrás.

In [ ]:
MEJORA = 0.12                     # la misma en los dos casos: es el mismo modelo
CONSTRUIR, MANTENER = 9000, 3600

sin_decision = valor_anual(decisiones_al_anio=0, mejora_esperada=MEJORA,
                           valor_unitario=MARGEN_ANUAL_CLIENTE,
                           costo_construir=CONSTRUIR, costo_mantener_anio=MANTENER)
con_decision = valor_anual(decisiones_al_anio=300, mejora_esperada=MEJORA,
                           valor_unitario=MARGEN_ANUAL_CLIENTE,
                           costo_construir=CONSTRUIR, costo_mantener_anio=MANTENER)

print("EL MISMO MODELO, CON LA MISMA EXACTITUD, EN DOS EMPRESAS DISTINTAS\n")
print(pd.DataFrame({
    "sin decisión detrás": sin_decision,
    "con presupuesto para 300 llamadas": con_decision,
}).to_string())

print(f"\ndiferencia en dinero: de {sin_decision['valor anual']:,.2f} "
      f"a {con_decision['valor anual']:,.2f} según haya o no una decisión")
print("  ← el número que decide si el proyecto existe")
print("\nBusca la palabra «exactitud» en la ecuación de valor: no aparece por ningún lado.")

📌 **El mismo modelo vale −6 600,00 o +3 577,21 al año.** No cambió el algoritmo, ni las variables, ni
la exactitud: cambió que alguien tenga presupuesto para llamar a 300 clientes y la instrucción de
hacerlo. Cuando el término de las decisiones es cero, la ecuación devuelve el costo del proyecto con
signo negativo.

La versión operativa de la trampa, y la única pregunta que hay que hacer en la primera reunión de
cualquier proyecto de datos:

> **«Cuando el modelo esté listo, ¿quién hace qué distinto el lunes por la mañana?»**

Si no hay respuesta con nombre, acción y frecuencia, el proyecto no está listo para empezar por mucho
que los datos estén perfectos. Y si la respuesta es «tendremos más visibilidad», la respuesta es que no.

Las tres preguntas que ordenan las semanas 12 a 15, en este orden: **¿qué se decide?** (verbo, sujeto y
frecuencia), **¿cuál es la línea base?** (calculada, no descrita) y **¿cuánto vale ganarle?** (con la
ecuación y los supuestos escritos). El algoritmo es la cuarta pregunta, y la más fácil.

## Entregable

Evaluación intermedia de criterio de negocio. Sube `lab_11_apellido.ipynb` con:

- El **catálogo de doce problemas completo**. La celda `verificar_catalogo()` tiene que imprimir
  **CATÁLOGO LISTO PARA ENTREGAR**.
- Los **problemas que no justifican ningún modelo**, al menos dos, demostrados con la ecuación de valor
  y no con opinión, cada uno con la frase de qué se hace en su lugar: un reporte, una regla, un
  experimento o ir a buscar el dato que falta.
- La **línea base calculada** del problema del proyecto propio, con `DummyClassifier` o
  `DummyRegressor` y el conjunto de prueba apartado con `train_test_split`.
- La **ecuación de valor** del problema propio con sus cinco supuestos y el punto de equilibrio.
- Una fila nueva en la bitácora de prompts: le pediste al asistente que resolviera tres de los doce
  problemas. **Cuenta cuántas veces propuso un modelo donde bastaba una regla** y anota el número. Es la
  medición del sesgo de la herramienta, y es parte de la entrega.

## Para tu equipo

- Las decisiones marcadas con ⚠️ en el blueprint de la semana pasada son la materia prima del catálogo:
  cada decisión sin evidencia es un candidato a problema.
- El problema que elijan para las semanas 12 a 15 tiene que tener **valor anual positivo con supuestos
  conservadores**. Si solo sale positivo con el supuesto optimista, elijan otro.
- Lleven la ecuación de valor a la conversación con la empresa del caso. La pregunta *«¿cuántas veces al
  año toman esta decisión?»* suele ser la primera que nadie ha contestado nunca, y la respuesta cambia
  el proyecto más que cualquier elección de algoritmo.